# 二吸収帯整合型 SC-LMMF のための合成メタンプルーム注入・評価

HISUIの実観測背景スペクトルに既知濃度の合成メタンプルームを注入し、1.6 µm帯と2.3 µm帯の両方を用いる推定手法を評価する。

合成プルーム注入によって、二吸収帯共通濃度の非線形LUTフィッティングが機能するかを評価するためのOracle-backgroundプロトタイプ。

## 評価の考え方

合成プルーム注入前の観測cubeを $L_{\mathrm{bg}}$、MODTRAN LUTから求めた濃度増分 $\alpha$ に対する放射輝度比を $T(\lambda,\alpha)$ とする。

合成プルームを注入したスペクトルは、次式で生成する。

$$
L_{\mathrm{syn}}(x,y,\lambda)
=
L_{\mathrm{bg}}(x,y,\lambda)
T\left(
\lambda,
\alpha_{\mathrm{true}}(x,y)
\right)
$$

ここで、

- $L_{\mathrm{syn}}(x,y,\lambda)$：合成プルーム注入後の放射輝度
- $L_{\mathrm{bg}}(x,y,\lambda)$：注入前の背景放射輝度
- $\alpha_{\mathrm{true}}(x,y)$：注入するメタン濃度増分の真値
- $T(\lambda,\alpha)$：MODTRAN LUTから求めた放射輝度比

である。

放射輝度比は、0 ppm増分のMODTRANスペクトルを基準として次式で求める。

$$
T(\lambda,\alpha)
=
\frac{
L_{\mathrm{MODTRAN}}(\lambda,\alpha)
}{
L_{\mathrm{MODTRAN}}(\lambda,0)
}
$$

この比を用いる場合、MODTRAN放射輝度を100倍してHISUIと単位を合わせても、分子と分母の両方に同じ係数が掛かるため、その係数は相殺される。

したがって、合成プルーム注入の処理では、MODTRAN放射輝度を100倍する必要はない。

`CH4b.csv` の各列名が背景濃度からのメタン濃度増分をppm単位で表している場合、$\alpha_{\mathrm{true}}$ と推定値 $\hat{\alpha}$ の単位もppmとなる。

## 重要な評価段階

### Stage A：Oracle background評価

注入前の観測cubeを真の背景スペクトルとして使用する。

この評価では、背景推定誤差を含めずに、次の要素を確認する。

- 合成プルーム注入処理
- MODTRAN LUT
- UAS
- 1.6 µm帯と2.3 µm帯の推定精度
- 二吸収帯融合
- 非線形フィッティング

### Stage B：Blind background評価

注入前の真の背景スペクトルを直接使用せず、Iterative MFやSVD低ランク再構成などによって背景スペクトルを推定する。

この評価では、実運用に近い条件で次の性能を確認する。

- 背景推定誤差の影響
- 地表面スペクトル変動への頑健性
- プルーム画素による背景統計量の汚染
- 偽陽性の発生
- 濃度推定精度

まずStage Aにおいて、既知の $\alpha_{\mathrm{true}}$ が正しく回収できることを確認する。

その後、真の背景スペクトルを既存のIterative MFやSVD背景再構成による推定背景へ置き換え、Stage Bの評価を行う。

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares

np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
ROI_CSV = 
CH4_LUT_CSV = 

FWHM_NM = 

WINDOW_16 = ()
WINDOW_23 = ()

UAS_PPM_RANGE = (0.0, 0.5)

PLUME_PEAK_PPM = 0.8
PLUME_CENTER_YX = None
PLUME_ANGLE_DEG = 20.0
PLUME_DECAY_PIX = 18.0
PLUME_CROSS_SIGMA_PIX = 4.0
PLUME_SOURCE_SIGMA_PIX = 2.0

CLIP_PPM_TO_LUT = True

## 2. HISUI ROI スペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    # wave_405.00nm のような列名から波長列を抽出する
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []

    for col in df.columns:
        match = pattern.match(str(col))
        if match:
            pairs.append((col, float(match.group(1))))

    if not pairs:
        raise ValueError(
            "wave_***nm形式の列が見つかりません。"
            f" 先頭列: {list(df.columns[:10])}"
        )

    pairs.sort(key=lambda item: item[1])
    wave_cols = [item[0] for item in pairs]
    wavelengths = np.array([item[1] for item in pairs], dtype=float)
    return wave_cols, wavelengths


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)

    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")

    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    ys = np.sort(df["y"].unique())
    xs = np.sort(df["x"].unique())

    y_to_i = {value: index for index, value in enumerate(ys)}
    x_to_j = {value: index for index, value in enumerate(xs)}

    cube = np.full(
        (len(ys), len(xs), spectra.shape[1]),
        fill_value,
        dtype=float
    )

    for row_number, row in df.reset_index(drop=True).iterrows():
        iy = y_to_i[row["y"]]
        ix = x_to_j[row["x"]]
        cube[iy, ix] = spectra[row_number]

    return cube, ys, xs


def make_valid_pixel_mask(
    cube,
    nodata_values=(0.0, -9999.0),
    require_positive=True,
    min_valid_fraction=1.0
):
    valid_band = np.isfinite(cube)

    for value in nodata_values:
        valid_band &= cube != value

    if require_positive:
        valid_band &= cube > 0

    return valid_band.mean(axis=2) >= min_valid_fraction


In [ ]:
df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_bg, ys, xs = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_bg)

print("DataFrame:", df.shape)
print("Cube:", cube_bg.shape)
print("Wavelength:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)
